# CCTV Safety — Stage 1 YOLOv8 training (Colab)

> **Historical training workflow:** Stage 1 YOLOv8n/YOLOv8s pilots were later run on Kaggle; see `docs/project_status.md`. This Colab notebook remains a reproduction workflow and will not train unless you deliberately change `TRAINING_APPROVED` to `True`. The unified dataset is provisional; review the validation report and known Fall / near-duplicate warnings before any new run.

The notebook trains YOLOv8n and YOLOv8s sequentially on the same six-class dataset. It writes runs and checkpoints to Google Drive. It does not train Fight; Fight belongs to Stage 2.

In [ ]:
from pathlib import Path
import json, shutil, zipfile, hashlib, csv, time

from google.colab import drive
drive.mount('/content/drive')

# Upload stage1_unified_colab.zip to this Drive folder (create it if needed).
DRIVE_ROOT = Path('/content/drive/MyDrive/CCTV-Safety-Stage1')
ARCHIVE_PATH = DRIVE_ROOT / 'stage1_unified_colab.zip'
DATA_ROOT = Path('/content/stage1_unified')
RUNS_ROOT = DRIVE_ROOT / 'training_runs'
# Keep this tag unchanged when reconnecting so the notebook finds the same checkpoints.
RUN_TAG = 'stage1_first_training'

assert ARCHIVE_PATH.is_file(), f'Upload the dataset ZIP to: {ARCHIVE_PATH}'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print('Archive size GiB:', round(ARCHIVE_PATH.stat().st_size / 1024**3, 2))
print('Drive output:', RUNS_ROOT / RUN_TAG)

In [ ]:
# Use Colab's selected GPU (the user reported T4 is available).
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Stop: select a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Extract/recover the dataset. A manifest alone is not proof of a complete prior extraction.
required_files = [DATA_ROOT / 'dataset_manifest.json', DATA_ROOT / 'data.yaml'] + [
    DATA_ROOT / 'metadata' / f'{s}.csv' for s in ('train', 'val', 'test')
]
EXTRACTION_MARKER = DATA_ROOT / '.extraction_complete'
archive_fingerprint = f'{ARCHIVE_PATH.stat().st_size}:{ARCHIVE_PATH.stat().st_mtime_ns}'
marker_matches = EXTRACTION_MARKER.is_file() and EXTRACTION_MARKER.read_text(encoding='utf-8').strip() == archive_fingerprint
if not all(p.is_file() for p in required_files) or not marker_matches:
    print('Dataset extraction missing required files; extracting/recovering from ZIP...')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_PATH) as zf:
        # Reading each member through ZipExtFile checks its CRC during extraction.
        root = DATA_ROOT.resolve()
        for member in zf.infolist():
            # Normalize Windows ZIP separators for Colab/Linux, then prevent path traversal.
            rel = member.filename.replace('\\', '/')
            target = (root / rel).resolve()
            assert target == root or root in target.parents, f'Unsafe ZIP path: {member.filename}'
            if member.is_dir() or rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
assert all(p.is_file() for p in required_files), 'Dataset extraction is still incomplete.'
EXTRACTION_MARKER.write_text(archive_fingerprint, encoding='utf-8')

manifest = json.loads((DATA_ROOT / 'dataset_manifest.json').read_text(encoding='utf-8'))
expected = ['person', 'helmet', 'vest', 'fall', 'fire', 'smoke']
assert manifest['class_names'] == expected, manifest['class_names']
assert manifest.get('training_approved') is False, 'Unexpected manifest approval state'
print('Images:', manifest['total_images'], '| instances:', manifest['total_instances'])
print('Schema:', manifest['class_names'])
print('Provisional / validation-only:', manifest.get('provisional'), manifest.get('validation_only'))
print('Training approved in manifest:', manifest['training_approved'])

In [ ]:
# Create an absolute-path dataset YAML for this Colab runtime.
import yaml
data_yaml = DATA_ROOT / 'data_colab.yaml'
data_cfg = {
    'path': str(DATA_ROOT.resolve()),
    'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
    'nc': 6, 'names': expected,
}
data_yaml.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding='utf-8')

# Pairing, class-range, and group-leakage preflight.
splits = ('train', 'val', 'test')
groups = {}
image_counts = {}
class_counts = {name: 0 for name in expected}
boundary_warnings = 0
for split in splits:
    imgs = list((DATA_ROOT / 'images' / split).glob('*'))
    imgs = [p for p in imgs if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}]
    labels = DATA_ROOT / 'labels' / split
    image_stems = {p.stem for p in imgs}
    label_stems = {p.stem for p in labels.glob('*.txt')}
    assert image_stems == label_stems, f'Image/label mismatch in {split}: missing labels={sorted(image_stems-label_stems)[:5]}, orphan labels={sorted(label_stems-image_stems)[:5]}'
    image_counts[split] = len(imgs)
    meta = DATA_ROOT / 'metadata' / f'{split}.csv'
    with meta.open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == len(imgs), f'Metadata row count mismatch in {split}'
    assert {r['image'] for r in rows} == {p.name for p in imgs}, f'Metadata/image filename mismatch in {split}'
    assert all(r['group_id'].strip() for r in rows), f'Empty group_id in {split}'
    groups[split] = {r['group_id'] for r in rows}
    for img in imgs:
        for line_no, line in enumerate((labels / f'{img.stem}.txt').read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip(): continue
            parts = line.split(); assert len(parts) == 5, f'{img.name}:{line_no}: bad YOLO row'
            cid = int(parts[0]); assert 0 <= cid < 6, f'{img.name}:{line_no}: class {cid}'
            class_counts[expected[cid]] += 1
            x,y,w,h = map(float, parts[1:]); assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
            if x-w/2 < 0 or y-h/2 < 0 or x+w/2 > 1 or y+h/2 > 1: boundary_warnings += 1
for a,b in [('train','val'),('train','test'),('val','test')]:
    assert not (groups[a] & groups[b]), f'Group leakage: {a}/{b}'
assert image_counts == manifest['images_by_split'], (image_counts, manifest['images_by_split'])
assert class_counts == manifest['instances'], (class_counts, manifest['instances'])
assert sum(class_counts.values()) == manifest['total_instances'], class_counts
print('Preflight PASS:', image_counts, '| class instances:', class_counts, '| boundary warnings:', boundary_warnings, '| data YAML:', data_yaml)

## Training approval guard

Do not change this until you have reviewed the provisional validation report and explicitly decided to start training. This guard is deliberately off by default.

In [ ]:
TRAINING_APPROVED = False  # Change to True only after your explicit owner sign-off.
if not TRAINING_APPROVED:
    raise RuntimeError('Training is disabled. Review the provisional dataset risks and explicitly set TRAINING_APPROVED=True to proceed.')

%pip -q install ultralytics==8.4.163
from ultralytics import YOLO
import ultralytics
print('Ultralytics:', ultralytics.__version__)

In [ ]:
# Runs sequentially; save_dir outputs are stored on Google Drive for persistence.
# If runtime stops, rerun this cell: unfinished models resume from Drive's last.pt.
# Conservative starting batch for the Colab T4 (15 GB VRAM shown in the user's runtime).
from pathlib import Path
import json

run_dir = RUNS_ROOT / RUN_TAG
run_dir.mkdir(parents=True, exist_ok=True)
model_results = {}
for model_name in ('yolov8n.pt', 'yolov8s.pt'):
    name = Path(model_name).stem
    model_dir = run_dir / name
    metrics_path = run_dir / f'{name}_test_metrics.json'
    training_done = model_dir / 'TRAINING_COMPLETE'
    if metrics_path.is_file():
        model_results[name] = json.loads(metrics_path.read_text(encoding='utf-8'))
        print(f'{name}: already finished; keeping saved Drive results.')
        continue
    best_path = run_dir / name / 'weights' / 'best.pt'
    last_path = run_dir / name / 'weights' / 'last.pt'
    if not training_done.is_file():
        if last_path.is_file():
            print(f'{name}: resuming from {last_path}')
            model = YOLO(str(last_path))
            model.train(resume=True)
        else:
            print('{}: starting new run; Drive checkpoint dir: {}'.format(name, model_dir / 'weights'))
            model = YOLO(model_name)
            model.train(
                data=str(data_yaml), epochs=100, imgsz=640, batch=8, patience=30, seed=42,
                workers=2, device=0, project=str(run_dir), name=name, exist_ok=True,
                save=True, save_period=-1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                fliplr=0.5, mosaic=1.0,
            )
    else:
        print(f'{name}: training already completed; continuing to test evaluation if needed.')
    assert best_path.is_file(), f'Missing best checkpoint: {best_path}'
    training_done.write_text('Training loop returned successfully.', encoding='utf-8')
    best = YOLO(str(best_path))
    test_metrics = best.val(data=str(data_yaml), split='test', device=0)
    result = {
        'model': name, 'best_weights': str(best_path),
        'map50': float(test_metrics.box.map50), 'map50_95': float(test_metrics.box.map),
        'precision': float(test_metrics.box.mp), 'recall': float(test_metrics.box.mr),
        'class_names': list(best.names.values()), 'dataset_manifest': str(DATA_ROOT / 'dataset_manifest.json'),
        'provisional_dataset': True,
    }
    (run_dir / f'{name}_test_metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    model_results[name] = result
(run_dir / 'comparison_summary.json').write_text(json.dumps(model_results, indent=2), encoding='utf-8')
print('Finished. Outputs:', run_dir)